<a href="https://colab.research.google.com/github/francescafarinaccio/apprendimento_automatico_e_apprendimento_profondo/blob/main/francesca_farinaccio_0322500078/WINE_QUALITY_CLASSIFIER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**WINE QUALITY CLASSIFIER**

1. SETUP E PRE-PROCESSING DEL DATASET "WINE QUALITY"

In [ ]:
!pip install ucimlrepo #installo la libreria per scaricare i dati da UCI via api
from ucimlrepo import fetch_ucirepo #funzione per prelevare il dataset dal web
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#scarico il dataset utilizzando il suo id univoco
wine_quality = fetch_ucirepo(id=186)

#separazione in features e target (in UCI sono già divisi)
X = wine_quality.data.features
Y = wine_quality.data.targets

print("METADATI")
print(wine_quality.metadata)

#controllo dei valori nulli
print("Valori mancanti per colonna:")
print(X.isnull().sum())

#suddivido il dataset in train e test
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

print("\nDimensioni del Train set:", X_train.shape)
print("Dimensioni del Test set:", X_test.shape)
print("\nTarget:", Y.value_counts().sort_index())

#standardizzazione - sottrae la media e divide per la deviaz standard
scaler = StandardScaler()
#calcolca i parametr e li applica al set di train
X_train_scaled = scaler.fit_transform(X_train)
#i dati di test vengono solo standardizzati ma non fittati
X_test_scaled = scaler.transform(X_test)


2. ANALISI PCA

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

#istanzio l'oggetto PCA
pca = PCA()

X_train_pca = pca.fit_transform(X_train_scaled)

varianza_singola = pca.explained_variance_ratio_
somma_varianze = np.cumsum(varianza_singola)

#creo lo scree plot per analizzare la curva in cui scende la varianza e dopo la quale le features sono meno significative
plt.figure(figsize=(10, 5))
plt.bar(range(1, len(varianza_singola) + 1), varianza_singola, alpha=0.6, align='center', label='Varianza singola componente', color='purple')
plt.step(range(1, len(somma_varianze) + 1), somma_varianze, where='mid', label='Varianza cumulata totale', color='red', linestyle='--')

plt.xlabel('Perc. varianza spiegata')
plt.ylabel('Varianza - Indice componenti PCA')
plt.title('SCREE PLOT - Analisi PCA')
plt.xticks(range(1, len(varianza_singola) + 1))
plt.legend(loc='best')
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()

#stampo anche i valori numerici
for i, var in enumerate(varianza_singola):
    print(f"PC{i+1}: spiega il {var*100:.2f}% della varianza")


3.1 Algoritmo di classificazione KNN

In [ ]:
import warnings
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import cross_val_score

# creo un nuovo oggetto PCA fissando n_components=3
pca_3 = PCA(n_components=3)

# trasformo dati scalati in sole 3 dimensioni
X_train_pca3 = pca_3.fit_transform(X_train_scaled)
X_test_pca3 = pca_3.transform(X_test_scaled)
print(f"Nuova forma di X_train dopo la PCA: {X_train_pca3.shape}")

# applico la cross validation per trovare il k ideale
k_range = range(1, 20)
k_scores = []

# filtro gli UserWarning per evitare messaggi sulle classi poco popolate
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=UserWarning)
    for k in k_range:
        knn_cv = KNeighborsClassifier(n_neighbors=k)
        scores = cross_val_score(knn_cv, X_train_pca3, Y_train.values.ravel(), cv=10, scoring='accuracy')
        k_scores.append(scores.mean())

optimal_k = k_range[np.argmax(k_scores)]
print(f"Il valore ottimale è: {optimal_k}\n")

# addestramento del modello KNN con il k ottimale
knn_model = KNeighborsClassifier(n_neighbors=optimal_k)

# addestro il modello usando i dati ridotti dalla PCA e i target corrispondenti - funzione ravel esegue il flattening trasformando y_train in vettore
knn_model.fit(X_train_pca3, Y_train.values.ravel())

# predizione sul test set
y_pred_knn = knn_model.predict(X_test_pca3)

# Calcoliamo l'accuratezza generale
accuratezza = accuracy_score(Y_test, y_pred_knn)
print(f"\nAccuratezza del modello KNN: {accuratezza * 100:.2f}%\n")

# Report di classificazione
print("--- REPORT DI CLASSIFICAZIONE KNN ---")
print(classification_report(Y_test, y_pred_knn, zero_division=0))

3.2 Algoritmo di classificazione SVM (Support Vector Machine)

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# 1. Inizializziamo il modello SVM
svm_model = SVC(kernel='rbf', random_state=42, probability=True)

# 2. Addestramento del modello
svm_model.fit(X_train_pca3, Y_train.values.ravel())

# 3. Predizione e Valutazione
y_pred_svm = svm_model.predict(X_test_pca3)

accuratezza_svm = accuracy_score(Y_test, y_pred_svm)
print(f"Accuratezza del modello SVM: {accuratezza_svm * 100:.2f}%\n")

print("--- REPORT DI CLASSIFICAZIONE SVM ---")
print(classification_report(Y_test, y_pred_svm, zero_division=0))

3.3 Algoritmo di classificazione Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, accuracy_score

# 1. Inizializziamo il modello Naïve Bayes Gaussiano
nb_model = GaussianNB()

# 2. Addestramento del modello
nb_model.fit(X_train_pca3, Y_train.values.ravel())

# 3. Predizione e Valutazione
y_pred_nb = nb_model.predict(X_test_pca3)

accuratezza_nb = accuracy_score(Y_test, y_pred_nb)
print(f"Accuratezza del modello Naïve Bayes: {accuratezza_nb * 100:.2f}%\n")

print("--- REPORT DI CLASSIFICAZIONE NAÏVE BAYES ---")
print(classification_report(Y_test, y_pred_nb, zero_division=0))

4. Matrice di confusione e curva ROC AUT

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.svm import SVC


# 1. PREPARAZIONE PER ROC AUC MULTI-CLASSE

# Binarizziamo i target per poter calcolare la curva ROC in un contesto multi-classe
classi_uniche = np.sort(np.unique(Y_test))
Y_test_bin = label_binarize(Y_test, classes=classi_uniche)
n_classes = Y_test_bin.shape[1]

# Otteniamo i punteggi di probabilità o decisioni per ogni modello
y_score_knn = knn_model.predict_proba(X_test_pca3)
y_score_nb = nb_model.predict_proba(X_test_pca3)
y_score_svm = svm_model.predict_proba(X_test_pca3)

# Calcolo del punteggio ROC AUC complessivo (Macro)
roc_knn = roc_auc_score(Y_test, y_score_knn, multi_class='ovr', average='macro')
roc_nb = roc_auc_score(Y_test, y_score_nb, multi_class='ovr', average='macro')
roc_svm = roc_auc_score(Y_test, y_score_svm, multi_class='ovr', average='macro')

print(f"ROC AUC Score (Macro) - KNN: {roc_knn:.4f}")
print(f"ROC AUC Score (Macro) - Naïve Bayes: {roc_nb:.4f}")
print(f"ROC AUC Score (Macro) - SVM: {roc_svm:.4f}\n")


# 2. VISUALIZZAZIONE DELLE MATRICI DI CONFUSIONE

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

ConfusionMatrixDisplay.from_predictions(Y_test, y_pred_knn, ax=axes[0], cmap='Blues')
axes[0].set_title('Matrice di Confusione: KNN')

ConfusionMatrixDisplay.from_predictions(Y_test, y_pred_nb, ax=axes[1], cmap='Greens')
axes[1].set_title('Matrice di Confusione: Naïve Bayes')

ConfusionMatrixDisplay.from_predictions(Y_test, y_pred_svm, ax=axes[2], cmap='Purples')
axes[2].set_title('Matrice di Confusione: SVM')

plt.tight_layout()
plt.show()

# 3. COSTRUZIONE E VISUALIZZAZIONE CURVA ROC MACRO
def calcola_macro_roc(Y_test_bin, y_score):
    fpr = dict()
    tpr = dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(Y_test_bin[:, i], y_score[:, i])
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    return all_fpr, mean_tpr, auc(all_fpr, mean_tpr)

fpr_knn, tpr_knn, auc_macro_knn = calcola_macro_roc(Y_test_bin, y_score_knn)
fpr_nb, tpr_nb, auc_macro_nb = calcola_macro_roc(Y_test_bin, y_score_nb)
fpr_svm, tpr_svm, auc_macro_svm = calcola_macro_roc(Y_test_bin, y_score_svm)

plt.figure(figsize=(8, 6))
plt.plot(fpr_knn, tpr_knn, label=f'KNN (AUC Macro = {auc_macro_knn:.2f})', color='blue', lw=2)
plt.plot(fpr_nb, tpr_nb, label=f'Naïve Bayes (AUC Macro = {auc_macro_nb:.2f})', color='green', lw=2)
plt.plot(fpr_svm, tpr_svm, label=f'SVM (AUC Macro = {auc_macro_svm:.2f})', color='purple', lw=2)
plt.plot([0, 1], [0, 1], 'k--', linestyle=':', label='Classificatore Casuale (AUC = 0.50)')

plt.xlabel('Tasso di Falsi Positivi (FPR)')
plt.ylabel('Tasso di Veri Positivi (TPR)')
plt.title('Confronto Curve ROC AUC (Macro-Average)')
plt.legend(loc='lower right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

Spiegabilità con modello SHAP

In [ ]:
# Installo direttamente nella cella
!pip install shap -q
import shap
import matplotlib.pyplot as plt
import numpy as np # Import numpy

# 1. Utilizziamo K-Means per creare un set di background compresso
# Questo evita il congelamento della sessione dovuto alla complessità matematica di SHAP
background_summary = shap.kmeans(X_train_pca3, 10)

# 2. Inizializziamo il KernelExplainer sul modello probabilistico Naive Bayes
explainer = shap.KernelExplainer(nb_model.predict_proba, background_summary)

# 3. Calcoliamo i valori SHAP su un campione controllato del Test Set (es. primi 40 campioni)
campionamento_test = X_test_pca3[:40]
shap_values = explainer.shap_values(campionamento_test)

# Aggreghiamo i valori SHAP per tutte le classi per ottenere un'importanza globale
# Se shap_values è un array 3D (samples, features, classes), sommiamo sull'asse delle classi (axis=2).
shap_values_aggregated = np.array(shap_values).sum(axis=2)

# 4. Generiamo il Summary Plot globale
# Mostra l'importanza delle 3 componenti della PCA aggregata su tutte le classi di vino
plt.figure(figsize=(8, 5))
shap.summary_plot(shap_values_aggregated, campionamento_test, feature_names=['PC1', 'PC2', 'PC3'], show=False)
plt.title("SHAP Feature Importance (Impatto globale di PC1, PC2, PC3)", fontsize=14, pad=20)
plt.show()